# 🏥 Clinical Survival Analysis
**Author:** Tharun Kumar Reddy Byreddy

**Goal:** Predict patient survival outcomes using Cox Proportional Hazards modeling

**Dataset:** Synthetic clinical registry data (500+ patients)

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

## 2. Load and Explore Data

In [ ]:
# Generate synthetic clinical data
np.random.seed(42)
n = 500

df = pd.DataFrame({
    'patient_id': range(1, n+1),
    'age': np.random.randint(30, 85, n),
    'gender': np.random.choice(['Male', 'Female'], n),
    'diagnosis': np.random.choice(['Type A', 'Type B', 'Type C'], n),
    'comorbidities': np.random.randint(0, 5, n),
    'treatment': np.random.choice(['Treatment 1', 'Treatment 2', 'Control'], n),
    'duration': np.random.exponential(365, n).astype(int),
    'event_observed': np.random.choice([0, 1], n, p=[0.4, 0.6])
})

df['risk_group'] = pd.cut(df['age'],
                           bins=[0, 50, 65, 100],
                           labels=['Low Risk', 'Medium Risk', 'High Risk'])

print('Dataset Shape:', df.shape)
df.head(10)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print('=== Dataset Info ===')
print(df.info())
print('\n=== Summary Statistics ===')
df.describe()

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Age distribution
axes[0,0].hist(df['age'], bins=20, color='steelblue', edgecolor='white')
axes[0,0].set_title('Age Distribution')
axes[0,0].set_xlabel('Age')

# Gender distribution
df['gender'].value_counts().plot(kind='bar', ax=axes[0,1], color=['steelblue','coral'])
axes[0,1].set_title('Gender Distribution')
axes[0,1].tick_params(rotation=0)

# Diagnosis distribution
df['diagnosis'].value_counts().plot(kind='bar', ax=axes[0,2], color='steelblue')
axes[0,2].set_title('Diagnosis Distribution')
axes[0,2].tick_params(rotation=0)

# Duration distribution
axes[1,0].hist(df['duration'], bins=30, color='coral', edgecolor='white')
axes[1,0].set_title('Survival Duration Distribution')
axes[1,0].set_xlabel('Days')

# Event observed
df['event_observed'].value_counts().plot(kind='pie', ax=axes[1,1],
                                          labels=['Censored','Event'],
                                          autopct='%1.1f%%',
                                          colors=['steelblue','coral'])
axes[1,1].set_title('Event Observed')

# Risk group
df['risk_group'].value_counts().plot(kind='bar', ax=axes[1,2],
                                      color=['green','orange','red'])
axes[1,2].set_title('Risk Group Distribution')
axes[1,2].tick_params(rotation=0)

plt.tight_layout()
plt.savefig('../results/eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA plot saved to results folder!')

## 4. Kaplan-Meier Survival Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KM by Risk Group
kmf = KaplanMeierFitter()
colors = {'Low Risk': 'green', 'Medium Risk': 'orange', 'High Risk': 'red'}

for group in df['risk_group'].dropna().unique():
    mask = df['risk_group'] == group
    kmf.fit(df[mask]['duration'],
            df[mask]['event_observed'],
            label=str(group))
    kmf.plot_survival_function(ax=axes[0],
                                color=colors[str(group)],
                                ci_show=True)

axes[0].set_title('Survival Curves by Risk Group')
axes[0].set_xlabel('Time (days)')
axes[0].set_ylabel('Survival Probability')

# KM by Treatment
treat_colors = {'Treatment 1': 'blue', 'Treatment 2': 'purple', 'Control': 'gray'}
for treatment in df['treatment'].unique():
    mask = df['treatment'] == treatment
    kmf.fit(df[mask]['duration'],
            df[mask]['event_observed'],
            label=treatment)
    kmf.plot_survival_function(ax=axes[1],
                                color=treat_colors[treatment],
                                ci_show=True)

axes[1].set_title('Survival Curves by Treatment')
axes[1].set_xlabel('Time (days)')
axes[1].set_ylabel('Survival Probability')

plt.tight_layout()
plt.savefig('../results/kaplan_meier_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('KM curves saved!')

## 5. Log-Rank Test

In [ ]:
# Log-rank test between Low Risk and High Risk
low_risk = df[df['risk_group'] == 'Low Risk']
high_risk = df[df['risk_group'] == 'High Risk']

results = logrank_test(
    low_risk['duration'], high_risk['duration'],
    low_risk['event_observed'], high_risk['event_observed']
)

print('=== Log-Rank Test: Low Risk vs High Risk ===')
print(f'Test Statistic : {results.test_statistic:.4f}')
print(f'P-Value        : {results.p_value:.6f}')
print(f'Significant    : {results.p_value < 0.05}')

## 6. Cox Proportional Hazards Model

In [ ]:
# Prepare features
cox_df = df.copy()
le = LabelEncoder()
cox_df['gender_enc'] = le.fit_transform(cox_df['gender'])
cox_df['diagnosis_enc'] = le.fit_transform(cox_df['diagnosis'])
cox_df['treatment_enc'] = le.fit_transform(cox_df['treatment'])

cox_features = cox_df[[
    'duration', 'event_observed',
    'age', 'gender_enc', 'comorbidities',
    'diagnosis_enc', 'treatment_enc'
]]

# Fit Cox PH model
cph = CoxPHFitter()
cph.fit(cox_features, duration_col='duration', event_col='event_observed')

print('=== Cox PH Model Summary ===')
cph.print_summary()

In [ ]:
# Concordance Index
print(f'Concordance Index: {cph.concordance_index_:.4f}')
print('(> 0.7 indicates good model discrimination)')

# Hazard Ratios Plot
fig, ax = plt.subplots(figsize=(8, 4))
hazard_ratios = cph.summary['exp(coef)']
hazard_ratios.index = ['Age', 'Gender', 'Comorbidities', 'Diagnosis', 'Treatment']
colors = ['red' if hr > 1 else 'green' for hr in hazard_ratios]
hazard_ratios.plot(kind='barh', ax=ax, color=colors)
ax.axvline(x=1, color='black', linestyle='--', linewidth=1.5)
ax.set_title('Hazard Ratios by Feature (>1 = increased risk)')
ax.set_xlabel('Hazard Ratio')
plt.tight_layout()
plt.savefig('../results/hazard_ratios.png', dpi=150, bbox_inches='tight')
plt.show()
print('Hazard ratio plot saved!')

## 7. Results Summary

In [ ]:
print('=' * 50)
print('       CLINICAL SURVIVAL ANALYSIS RESULTS')
print('=' * 50)
print(f'Total Patients Analyzed  : {len(df)}')
print(f'Events Observed          : {df["event_observed"].sum()}')
print(f'Censored Patients        : {(df["event_observed"]==0).sum()}')
print(f'Median Survival Duration : {int(df["duration"].median())} days')
print(f'Cox PH Concordance Index : {cph.concordance_index_:.4f}')
print(f'Log-Rank P-Value         : {results.p_value:.6f}')
print('=' * 50)
print('Author: Tharun Kumar Reddy Byreddy')
print('M.S. Statistical Data Science | SFSU')